# Лекция: Описательные статистики в Python

**Дисциплина:** Введение в анализ больших данных

В этой лекции — базовые описательные статистики и простые графики для числовых и категориальных данных:
- среднее, стандартная ошибка, медиана, квантили;
- асимметрия и эксцесс;
- группировка (`groupby`);
- гистограмма с плотностью, boxplot, barplot.

Инструменты: **pandas**, **NumPy**, **SciPy**, **Matplotlib**, **Seaborn**.

Примеры построены на датасете **tips** (чаевые в ресторане). Они **не совпадают** с лабораторным заданием: цель — освоить методы, а само задание выполнить самостоятельно на своих данных.


## 0. Импорт и загрузка данных


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")

# Демо-данные: чаевые (seaborn)
tips = sns.load_dataset("tips")
print("Размерность:", tips.shape)
print("Столбцы:", tips.columns.tolist())
tips.head()


In [ ]:
tips.describe(include="all")


---
## 1. Гистограмма числовой переменной + кривая плотности

Для непрерывного показателя удобно совместить:
- гистограмму (`density=True` — нормировка на плотность);
- ядерную оценку плотности (KDE) или теоретическую кривую.

Ниже — распределение **суммы счёта** (`total_bill`).


In [ ]:
fig, ax = plt.subplots()
ax.hist(tips["total_bill"], bins=25, density=True, color="lightblue",
        edgecolor="black", alpha=0.7, label="гистограмма")

# KDE по данным
kde = stats.gaussian_kde(tips["total_bill"].dropna())
xs = np.linspace(tips["total_bill"].min(), tips["total_bill"].max(), 300)
ax.plot(xs, kde(xs), color="crimson", lw=2, label="KDE")

ax.set_xlabel("total_bill")
ax.set_ylabel("плотность")
ax.set_title("Гистограмма total_bill + KDE")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# То же через seaborn
plt.figure()
sns.histplot(tips["total_bill"], bins=25, kde=True, color="steelblue",
             edgecolor="black", alpha=0.7)
plt.xlabel("total_bill")
plt.title("Гистограмма + KDE (seaborn)")
plt.tight_layout()
plt.show()


---
## 2. Среднее и стандартная ошибка среднего (SE)

$$
\mathrm{SE} = \frac{s}{\sqrt{n}},
$$
где $s$ — выборочное стандартное отклонение, $n$ — объём выборки.


In [ ]:
x = tips["total_bill"]
n = x.count()
mean_x = x.mean()
sd_x = x.std(ddof=1)
se_x = sd_x / np.sqrt(n)

print(f"n     = {n}")
print(f"mean  = {mean_x:.4f}")
print(f"sd    = {sd_x:.4f}")
print(f"SE    = {se_x:.4f}")


---
## 3. Медиана, квантили, summary

`describe` / `quantile` дают «сжатый» портрет распределения.


In [ ]:
print("Медиана:", x.median())
print("Квантили 0%, 25%, 50%, 75%, 100%:")
print(x.quantile([0, 0.25, 0.5, 0.75, 1.0]))
print()
print(x.describe())


---
## 4. Асимметрия (skewness) и эксцесс (kurtosis)

- **Skewness** > 0 — правый хвост длиннее (типично для сумм платежей).
- **Kurtosis** (excess) > 0 — хвосты тяжелее, чем у нормального.

В SciPy: `stats.skew`, `stats.kurtosis` (по умолчанию *excess* kurtosis).


In [ ]:
sk = stats.skew(x, bias=False)
ku = stats.kurtosis(x, bias=False)
print(f"skewness = {sk:.4f}")
print(f"kurtosis (excess) = {ku:.4f}")


---
## 5. Описательные статистики **по группам**

`groupby` + агрегирующая функция — аналог сводных таблиц по фактору.


In [ ]:
# Сводка total_bill по полу клиента
print(tips.groupby("sex")["total_bill"].describe().round(2))


In [ ]:
# Несколько статистик сразу
tips.groupby("day")["tip"].agg(["count", "mean", "median", "std"]).round(3)


---
## 6. Boxplot по категориям

«Ящик с усами» показывает медиану, квартили и выбросы по группам.


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=tips, x="day", y="total_bill", hue="sex")
plt.title("total_bill по дням недели и полу")
plt.tight_layout()
plt.show()


---
## 7. Медианы в разрезе двух факторов

Сводная таблица медиан, затем столбчатая диаграмма.


In [ ]:
med_two = tips.groupby(["day", "sex"])["tip"].median().unstack()
print(med_two.round(2))

med_two.plot(kind="bar", figsize=(9, 5), edgecolor="black")
plt.ylabel("Медиана tip")
plt.xlabel("День")
plt.title("Медианные чаевые: день × пол")
plt.xticks(rotation=0)
plt.legend(title="sex")
plt.tight_layout()
plt.show()


---
## 8. Barplot для одной группировки

Сначала считаем агрегат, потом рисуем столбцы.


In [ ]:
med_by_time = tips.groupby("time")["total_bill"].median().sort_values(ascending=False)
print(med_by_time)

plt.figure(figsize=(6, 4))
med_by_time.plot(kind="bar", color="teal", edgecolor="black")
plt.ylabel("Медиана total_bill")
plt.title("Медианный счёт: обед vs ужин")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


---
## 9. Ещё один пример группировки: размер компании

Медиана чаевых по числу гостей (`size`) — вся выборка и отдельно по времени дня.


In [ ]:
# Вся выборка
med_size = tips.groupby("size")["tip"].median()
print("Медиана tip по size:")
print(med_size)

# По времени дня
med_size_time = tips.groupby(["time", "size"])["tip"].median().unstack()
print("\nМедиана tip: time × size")
print(med_size_time.round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

med_size.plot(kind="bar", ax=axes[0], color="coral", edgecolor="black")
axes[0].set_title("Медиана tip по size (вся выборка)")
axes[0].set_ylabel("Медиана tip")
axes[0].set_xlabel("size")

med_size_time.T.plot(kind="bar", ax=axes[1], edgecolor="black")
axes[1].set_title("Медиана tip: size × time")
axes[1].set_ylabel("Медиана tip")
axes[1].set_xlabel("size")
axes[1].legend(title="time")

for ax in axes:
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


---
## Шпаргалка по методам (Python)

| Задача | Код |
|--------|-----|
| Загрузка CSV | `pd.read_csv(path_or_url)` |
| Среднее / sd | `s.mean()`, `s.std(ddof=1)` |
| SE среднего | `s.std(ddof=1) / np.sqrt(s.count())` |
| Медиана, квантили | `s.median()`, `s.quantile([0.25, 0.5, 0.75])` |
| Summary | `s.describe()` / `df.describe()` |
| Skewness / kurtosis | `stats.skew(x)`, `stats.kurtosis(x)` |
| По группам | `df.groupby("A")["y"].mean()` / `.median()` / `.describe()` |
| Два фактора | `df.groupby(["A", "B"])["y"].median().unstack()` |
| Гистограмма + KDE | `plt.hist(..., density=True)` + KDE / `sns.histplot(..., kde=True)` |
| Boxplot | `sns.boxplot(data=df, x="A", y="y")` |
| Barplot | `series.plot(kind="bar")` |

---
## Что сделать после лекции

1. Повторите расчёты на **других** столбцах `tips` (или своём CSV).
2. Откройте лабораторное задание и выполните его **самостоятельно** на указанном там наборе данных.
3. Обращайте внимание на тип переменной: числовая vs категориальная — от этого зависит выбор графика и агрегата.

Удачи с описательными статистиками!
